# Phase 4 — Post Fine-Tuning Evaluation
### Compare Baseline vs Fine-Tuned Model

This notebook:
1. Loads the fine-tuned LoRA adapter
2. Runs same 200 validation samples as baseline
3. Compares before vs after scores
4. Runs MMLU subset to check catastrophic forgetting
5. Saves final comparison results


In [1]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("All imports successful!")
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0)}")


All imports successful!
GPU available : True
GPU name      : NVIDIA GeForce RTX 5060 Laptop GPU


## Step 1 — Configuration

In [2]:
CONFIG = {
    "base_model"        : "Qwen/Qwen1.5-1.8B-Chat",
    "adapter_path"      : "./outputs/qwen-sql-qlora/final_adapter",
    "data_dir"          : "./data/spider_formatted",
    "baseline_file"     : "./data/baseline_results.json",
    "output_file"       : "./data/finetuned_results.json",
    "comparison_file"   : "./data/comparison_results.json",
    "num_samples"       : 200,
    "max_new_tokens"    : 128,
    "seed"              : 42,
}

torch.manual_seed(CONFIG["seed"])
print("Config ready!")
print(f"Adapter path : {CONFIG['adapter_path']}")


Config ready!
Adapter path : ./outputs/qwen-sql-qlora/final_adapter


## Step 2 — Load Baseline Results

In [3]:
# Load baseline scores from Phase 2
with open(CONFIG["baseline_file"], "r") as f:
    baseline = json.load(f)

print("=== BASELINE SCORES (Before Fine-Tuning) ===")
print(f"  Model           : {baseline['model']}")
print(f"  Samples tested  : {baseline['samples_evaluated']}")
print(f"  Exact Match Acc : {baseline['exact_match_accuracy']}%")
print(f"  Avg Token Match : {baseline['avg_token_match']}%")
print("These are the scores we need to BEAT!")


=== BASELINE SCORES (Before Fine-Tuning) ===
  Model           : Qwen/Qwen1.5-1.8B-Chat
  Samples tested  : 500
  Exact Match Acc : 0.0%
  Avg Token Match : 34.39%
These are the scores we need to BEAT!


## Step 3 — Load Validation Dataset

In [4]:
dataset  = load_from_disk(CONFIG["data_dir"])
val_data = dataset["validation"]
val_subset = val_data.select(range(CONFIG["num_samples"]))

print(f"Validation samples loaded : {len(val_subset)}")
print(f"Sample question: {val_subset[0]['question']}")


Validation samples loaded : 200
Sample question: How many singers do we have?


## Step 4 — Load Fine-Tuned Model

In [5]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model in 4-bit
print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config = bnb_config,
    device_map          = {"": 0},
    trust_remote_code   = True,
)

# Load LoRA adapter on top
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, CONFIG["adapter_path"])
model.eval()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nFine-tuned model loaded!")
print(f"VRAM used : {vram_used:.2f} GB")


Loading tokenizer...
Loading base model in 4-bit...


W0529 23:34:18.546000 12516 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
d:\Project\sql_finetune_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading LoRA adapter...

Fine-tuned model loaded!
VRAM used : 2.16 GB


## Step 5 — Run Inference

In [6]:
def generate_sql(question, db_id, model, tokenizer, max_new_tokens=128):
    """Generate SQL using fine-tuned model."""
    prompt = f"""<|im_start|>system
You are an expert SQL assistant that converts natural language questions into accurate SQL queries.<|im_end|>
<|im_start|>user
You are an expert SQL assistant. Given a natural language question and a database name, write the correct SQL query.

Database: {db_id}
Question: {question}

Write only the SQL query, nothing else.<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            do_sample         = False,
            temperature       = 1.0,
            pad_token_id      = tokenizer.eos_token_id,
            eos_token_id      = tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
    sql = sql.split('<|im_end|>')[0].strip()
    sql = sql.split('\n\n')[0].strip()

    # ── NEW LINES ──────────────────────────────────────────
    # Stop at semicolon - take only first SQL statement
    if ';' in sql:
        sql = sql.split(';')[0].strip() + ';'
    # Remove non-ASCII characters (Chinese text, symbols etc.)
    sql = ''.join(char for char in sql if ord(char) < 128).strip()
    # Clean extra whitespace
    sql = ' '.join(sql.split())
    # ───────────────────────────────────────────────────────

    return sql

def normalize_sql(sql):
    sql = sql.lower().strip()
    sql = ' '.join(sql.split())
    sql = sql.rstrip(';').strip()
    return sql

def exact_match(predicted, expected):
    return normalize_sql(predicted) == normalize_sql(expected)

def token_match(predicted, expected):
    pred_tokens = set(normalize_sql(predicted).split())
    exp_tokens  = set(normalize_sql(expected).split())
    if not exp_tokens:
        return 0.0
    return len(pred_tokens.intersection(exp_tokens)) / len(exp_tokens)

# Test on one sample
print("Testing fine-tuned model on one sample...")
sample = val_subset[0]
generated = generate_sql(sample['question'], sample['db_id'], model, tokenizer)
print(f"Question : {sample['question']}")
print(f"Expected : {sample['query']}")
print(f"Got      : {generated}")
print(f"Match    : {exact_match(generated, sample['query'])}")

Testing fine-tuned model on one sample...


d:\Project\sql_finetune_env\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
d:\Project\sql_finetune_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Question : How many singers do we have?
Expected : SELECT count(*) FROM singer
Got      : SELECT count(*) FROM singer;
Match    : True


## Step 6 — Evaluate All 200 Samples

In [7]:
results      = []
exact_matches = 0
token_scores  = []

print(f"Evaluating {CONFIG['num_samples']} samples...")
print("This will take 10-20 minutes\n")

for i, sample in enumerate(tqdm(val_subset, desc="Evaluating")):
    try:
        generated = generate_sql(
            sample['question'],
            sample['db_id'],
            model,
            tokenizer,
            max_new_tokens=CONFIG['max_new_tokens']
        )

        em = exact_match(generated, sample['query'])
        tm = token_match(generated, sample['query'])

        if em:
            exact_matches += 1
        token_scores.append(tm)

        results.append({
            "id"            : i,
            "question"      : sample['question'],
            "db_id"         : sample['db_id'],
            "expected_sql"  : sample['query'],
            "generated_sql" : generated,
            "exact_match"   : em,
            "token_match"   : round(tm, 4),
        })

        if (i + 1) % 50 == 0:
            current_acc = exact_matches / (i + 1) * 100
            print(f"  [{i+1}/{CONFIG['num_samples']}] Exact match so far: {current_acc:.1f}%")

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        results.append({
            "id"            : i,
            "question"      : sample['question'],
            "db_id"         : sample['db_id'],
            "expected_sql"  : sample['query'],
            "generated_sql" : "ERROR",
            "exact_match"   : False,
            "token_match"   : 0.0,
        })

print("\nEvaluation complete!")


Evaluating 200 samples...
This will take 10-20 minutes



Evaluating:  25%|██▌       | 50/200 [03:59<10:31,  4.21s/it]

  [50/200] Exact match so far: 24.0%


Evaluating:  50%|█████     | 100/200 [07:43<07:22,  4.42s/it]

  [100/200] Exact match so far: 12.0%


Evaluating:  75%|███████▌  | 150/200 [12:27<03:32,  4.25s/it]

  [150/200] Exact match so far: 8.0%


Evaluating: 100%|██████████| 200/200 [15:37<00:00,  4.69s/it]

  [200/200] Exact match so far: 6.0%

Evaluation complete!


## Step 7 — Compare Before vs After

In [9]:
# Calculate fine-tuned scores
exact_match_acc = exact_matches / len(results) * 100
avg_token_match = sum(token_scores) / len(token_scores) * 100

# Load baseline for comparison
baseline_em = baseline['exact_match_accuracy']
baseline_tm = baseline['avg_token_match']

# Calculate improvement
em_improvement = exact_match_acc - baseline_em
tm_improvement = avg_token_match - baseline_tm

print("=" * 55)
print("         BEFORE vs AFTER COMPARISON")
print("=" * 55)
print(f"{'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>12} {'Change':>8}")
print("-" * 55)
print(f"{'Exact Match Acc':<25} {baseline_em:>9.2f}% {exact_match_acc:>11.2f}% {em_improvement:>+7.2f}%")
print(f"{'Avg Token Match':<25} {baseline_tm:>9.2f}% {avg_token_match:>11.2f}% {tm_improvement:>+7.2f}%")
print("=" * 55)

if exact_match_acc > baseline_em:
    print(f"\n✅ Exact match improved by {em_improvement:.2f} percentage points!")
else:
    print("\n⚠️  No improvement in exact match - check your results")

# Show correct predictions
print("\n--- Correct predictions (fine-tuned) ---")
correct = [r for r in results if r['exact_match']][:5]
for r in correct:
    print(f"  Q: {r['question']}")
    print(f"  Expected : {r['expected_sql']}")
    print(f"  Got      : {r['generated_sql']}")
    print()


         BEFORE vs AFTER COMPARISON
Metric                      Baseline   Fine-Tuned   Change
-------------------------------------------------------
Exact Match Acc                0.00%        6.00%   +6.00%
Avg Token Match               34.39%       54.75%  +20.36%

✅ Exact match improved by 6.00 percentage points!

--- Correct predictions (fine-tuned) ---
  Q: How many singers do we have?
  Expected : SELECT count(*) FROM singer
  Got      : SELECT count(*) FROM singer;

  Q: What are the names, countries, and ages for every singer in descending order of age?
  Expected : SELECT name ,  country ,  age FROM singer ORDER BY age DESC
  Got      : SELECT Name , Country , Age FROM singer ORDER BY Age DESC;

  Q: What is the average, minimum, and maximum age of all singers from France?
  Expected : SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
  Got      : SELECT avg(age) , min(age) , max(age) FROM singer WHERE country = 'France';

  Q: What are all disti

## Step 8 — Catastrophic Forgetting Check (MMLU)

In [10]:
# Quick MMLU check using 50 samples
# Tests if model still has general knowledge after fine-tuning
print("Running MMLU subset check (50 samples)...")
print("This checks if model forgot general knowledge after SQL training\n")

try:
    mmlu = load_dataset("cais/mmlu", "all", split="test[:50]")

    mmlu_correct = 0
    choices_map  = {0: "A", 1: "B", 2: "C", 3: "D"}

    for sample in tqdm(mmlu, desc="MMLU"):
        prompt = f"""<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{sample['question']}

A) {sample['choices'][0]}
B) {sample['choices'][1]}
C) {sample['choices'][2]}
D) {sample['choices'][3]}

Answer with only the letter (A, B, C, or D).<|im_end|>
<|im_start|>assistant
"""
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = 5,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )

        generated  = outputs[0][inputs['input_ids'].shape[1]:]
        answer     = tokenizer.decode(generated, skip_special_tokens=True).strip()
        first_char = answer[0].upper() if answer else "X"

        expected = choices_map.get(sample['answer'], "X")
        if first_char == expected:
            mmlu_correct += 1

    mmlu_acc = mmlu_correct / 50 * 100
    print(f"\nMMlu accuracy (50 samples): {mmlu_acc:.1f}%")
    print(f"Random baseline would be  : 25.0%")

    if mmlu_acc >= 25:
        print("✅ No catastrophic forgetting detected!")
    else:
        print("⚠️  Possible catastrophic forgetting - score below random baseline")

except Exception as e:
    mmlu_acc = None
    print(f"MMLU check skipped: {e}")
    print("You can run lm-evaluation-harness separately for MMLU")


Running MMLU subset check (50 samples)...
This checks if model forgot general knowledge after SQL training



Using the latest cached version of the dataset since cais/mmlu couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'all' at C:\Users\tejaa\.cache\huggingface\datasets\cais___mmlu\all\0.0.0\c30699e8356da336a370243923dbaf21066bb9fe (last modified on Fri May 29 15:42:13 2026).
MMLU:   0%|          | 0/50 [00:00<?, ?it/s]d:\Project\sql_finetune_env\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
d:\Project\sql_finetune_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
MMLU: 100%|██████████| 50/50 [00:20<00:00,  2.42it/s]


MMlu accuracy (50 samples): 16.0%
Random baseline would be  : 25.0%
⚠️  Possible catastrophic forgetting - score below random baseline


## Step 9 — Save All Results

In [11]:
# Save fine-tuned results
finetuned_summary = {
    "model"                : CONFIG["base_model"],
    "adapter"              : CONFIG["adapter_path"],
    "phase"                : "fine-tuned (after QLoRA training)",
    "samples_evaluated"    : len(results),
    "exact_match_accuracy" : round(exact_match_acc, 2),
    "avg_token_match"      : round(avg_token_match, 2),
    "detailed_results"     : results,
}

with open(CONFIG["output_file"], "w") as f:
    json.dump(finetuned_summary, f, indent=2)

# Save comparison
comparison = {
    "baseline": {
        "exact_match_accuracy" : baseline_em,
        "avg_token_match"      : baseline_tm,
    },
    "finetuned": {
        "exact_match_accuracy" : round(exact_match_acc, 2),
        "avg_token_match"      : round(avg_token_match, 2),
    },
    "improvement": {
        "exact_match_delta"    : round(em_improvement, 2),
        "token_match_delta"    : round(tm_improvement, 2),
    },
    "mmlu_accuracy"            : round(mmlu_acc, 2) if mmlu_acc else "not run",
}

with open(CONFIG["comparison_file"], "w") as f:
    json.dump(comparison, f, indent=2)

print(f"Results saved!")
print(f"  {CONFIG['output_file']}")
print(f"  {CONFIG['comparison_file']}")
print("\n" + "="*50)
print("  Phase 4 Complete!")
print("="*50)
print(f"  Baseline Exact Match  : {baseline_em}%")
print(f"  Fine-Tuned Exact Match: {round(exact_match_acc, 2)}%")
print(f"  Improvement           : +{round(em_improvement, 2)}%")
print("="*50)
print("\nNext: Phase 5 - Publish to Hugging Face Hub!")


Results saved!
  ./data/finetuned_results.json
  ./data/comparison_results.json

  Phase 4 Complete!
  Baseline Exact Match  : 0.0%
  Fine-Tuned Exact Match: 6.0%
  Improvement           : +6.0%

Next: Phase 5 - Publish to Hugging Face Hub!
